In [ ]:
# --- SPACEMAP notebook configuration ------------------------------------
# Every path below is built from DATA_DIR. Point it at your own data by
# setting SPACEMAP_DATA_DIR in your environment, or by editing the default.
import os
from pathlib import Path

DATA_DIR = os.environ.get("SPACEMAP_DATA_DIR", str(Path.home() / "spacemap-data"))
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
print(f"Using DATA_DIR = {DATA_DIR}")


# Background Subtraction for OME-TIFF Images

This notebook performs background subtraction on multi-channel OME-TIFF images. It is designed to remove noise from fluorescence channels by subtracting a corresponding 'blank' channel on a pixel-by-pixel basis.

## Workflow

1.  **Discovery**: The notebook first inspects the input OME-TIFF to identify all available channels.
2.  **Configuration**: You must define which channels are 'blanks' and which fluorescence markers correspond to each blank.
3.  **Processing**: The notebook loads the marker and blank channels, performs the subtraction (`new_pixel = max(0, marker - blank)`), and saves the result.
4.  **Output**: A new pyramidal OME-TIFF is created, preserving all original metadata and channel structures, with the background-subtracted data.

In [ ]:
# Imports and Environment Setup
import sys
import warnings
from pathlib import Path
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Add src to path for local imports
project_root = Path().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from spacemap.utils import log_time
from spacemap.background_subtraction_utils import (
    get_channel_info,
    get_channel_records,
    get_image_mpp,
    normalize_image_stem,
    read_channel_image,
    subtract_background_and_save
    )

print(f"Project root: {project_root}")
print("Setup Complete")

In [ ]:
# =============================================================================
# METADATA INSPECTION (Temporary)
# =============================================================================
import tifffile

# file_path = f"{DATA_DIR}/example_project/data/codex/20251117_HCC_opt1_Scan1.er.qptiff"
file_path = project_root / 'data' / 'codex' / '20251117_HCC_opt1_Scan1.er.qptiff'

print(f"Inspecting metadata for: {file_path}")

if file_path.exists():
    try:
        channel_records, detected_mpp = get_channel_records(str(file_path))
        print(f"\nDetected MPP: {detected_mpp.get('codex', 'Unknown')}")
        print(f"Detected {len(channel_records)} channels")

        if channel_records:
            print("\n--- Parsed Channel Metadata (First 10) ---")
            for i, record in enumerate(channel_records[:10], 1):
                meta = record['metadata']
                color_hex = '#{:02X}{:02X}{:02X}'.format(*record['color'])
                print(
                    f"{i}. {record['name']} | Color: {color_hex} | "
                    f"Dye: {meta.get('dye', '-')} | Exp: {meta.get('exposure', '-')} | "
                    f"Em: {meta.get('emission_wavelength', '-')} | Source: {meta.get('color_source', '-')}"
                )
            if len(channel_records) > 10:
                print(f"... and {len(channel_records) - 10} more channels")

        with tifffile.TiffFile(file_path) as tif:
            if tif.ome_metadata:
                print("\n--- OME-XML Metadata Found ---")
            else:
                print("\n--- No OME-XML Metadata (Expected for QPTIFF) ---")

            page = tif.pages[0]
            desc = getattr(page, 'description', '')
            if desc:
                print("\n--- Image Description Snippet ---")
                print(desc[:2000])
                if len(desc) > 2000:
                    print("... [truncated] ...")
            else:
                print("No Image Description found.")

    except Exception as e:
        print(f"Error reading file: {e}")
else:
    print("File not found.")

## 1. Configuration

Define the input image path, output path, and the mapping between blank channels and their corresponding marker channels.

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# --- REQUIRED: Define Input and Output Paths ---
INPUT_IMAGE_PATH = project_root / 'outputs' / 'registration' / 'merged' / 'RCC_merged.ome.tiff'
OUTPUT_DIR = project_root / 'outputs' / 'background_subtracted'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_filename = f"{normalize_image_stem(INPUT_IMAGE_PATH)}_bg_subtracted.ome.tiff"
OUTPUT_IMAGE_PATH = OUTPUT_DIR / output_filename

# --- REQUIRED: Define Blank-to-Marker Mapping ---
# Use the channel names discovered in the next step to fill this out.
BLANK_MAPPING = {
    'Blank6 Atto550': [
        'CD44',
        'E-cadherin',
        'CD45RO',
        'CD69',
        'CD8',
        'Ki67',
        'CD15',
        'GLUT1',
        'LAG3'
    ],
    'Blank6 Cy5': [
        'FoxP3',
        'CD4',
        'CD68',
        'CD11c',
        'HLA-DR',
        'CD3e',
        'PDGFRa',
        'CD163',
        'PD-1',
        'CD11b',
        'PD-L1',
        'CD40',
        'PCNA',
        'Granzyme B'
    ],
    'Blank6 AF750': [
        'CD31',
        'CD20',
        'Pan-Cytokeratin',
        'HLA-A',
        'CD45'
    ]
}

DETECTED_INPUT_MPP = get_image_mpp(str(INPUT_IMAGE_PATH)).get('codex', 0.5083) if INPUT_IMAGE_PATH.exists() else 0.5083

log_time("Configuration loaded.")
print(f"Input Image: {INPUT_IMAGE_PATH}")
print(f"Output Image: {OUTPUT_IMAGE_PATH}")
print(f"Detected input MPP: {DETECTED_INPUT_MPP:.4f}")

## 2. Discovery Phase

This step reads the metadata from the input OME-TIFF and lists all available channels. Use this list to verify your `BLANK_MAPPING` configuration in the cell above.

In [ ]:
# =============================================================================
# DISCOVERY
# =============================================================================
log_time("Starting discovery phase...")

if not INPUT_IMAGE_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_IMAGE_PATH}")

channel_names, channel_colors, channel_meta, mpp = get_channel_info(str(INPUT_IMAGE_PATH))

if channel_names:
    print("\n--- Available Channels ---")
    for i, name in enumerate(channel_names):
        print(f"{i+1:02d}: {name}")
    print("--------------------------\n")
    print(f"Detected MPP from metadata: {mpp.get('codex', DETECTED_INPUT_MPP):.4f}")

    all_mapped_markers = [marker for markers in BLANK_MAPPING.values() for marker in markers]
    all_blanks = list(BLANK_MAPPING.keys())

    missing_markers = [m for m in all_mapped_markers if m not in channel_names]
    missing_blanks = [b for b in all_blanks if b not in channel_names]

    if missing_markers:
        print(f"\nWARNING: The following mapped markers are not in the image: {missing_markers}")
    if missing_blanks:
        print(f"\nWARNING: The following blank channels are not in the image: {missing_blanks}")

    log_time("Discovery complete. Please verify your BLANK_MAPPING.")
else:
    log_time("Could not retrieve channel information. Please check the input file.")

## 3. Processing Phase

This is the main processing step. The code will:
1. Load each marker channel and its corresponding blank channel.
2. Perform pixel-wise subtraction.
3. Handle channels that do not require subtraction.
4. Create a new pyramidal OME-TIFF with the corrected data.

This process is memory-efficient and will take some time depending on the image size.

In [ ]:
# =============================================================================
# PROCESSING
# =============================================================================
log_time("Starting background subtraction process...")

if not channel_names:
    log_time("Cannot proceed without channel information. Halting execution.")
else:
    subtract_background_and_save(
        input_path=INPUT_IMAGE_PATH,
        output_path=OUTPUT_IMAGE_PATH,
        blank_mapping=BLANK_MAPPING,
        channel_names=channel_names,
        channel_colors=channel_colors,
        mpp=mpp
    )
    log_time("Processing complete.")

## 4. Verification (Optional)

To verify the subtraction, this step loads a sample marker channel from the original and the new image and displays a small region for comparison.

In [ ]:
# =============================================================================
# VERIFICATION
# =============================================================================
log_time("Starting verification...")

VERIFY_MARKER = 'CD45'  # A marker that was subtracted
CROP_COORDS = (1000, 1200, 4800, 5000)  # (y1, y2, x1, x2)

if OUTPUT_IMAGE_PATH.exists() and VERIFY_MARKER in channel_names:
    try:
        marker_idx = channel_names.index(VERIFY_MARKER)
        blank_name = [b for b, m in BLANK_MAPPING.items() if VERIFY_MARKER in m][0]
        blank_idx = channel_names.index(blank_name)
        
        y1, y2, x1, x2 = CROP_COORDS

        original_marker = read_channel_image(str(INPUT_IMAGE_PATH), marker_idx, region=(y1, y2, x1, x2))
        original_blank = read_channel_image(str(INPUT_IMAGE_PATH), blank_idx, region=(y1, y2, x1, x2))
        subtracted_marker = read_channel_image(str(OUTPUT_IMAGE_PATH), marker_idx, region=(y1, y2, x1, x2))

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        im1 = axes[0].imshow(original_marker, cmap='hot')
        axes[0].set_title(f'Original {VERIFY_MARKER}')
        axes[0].axis('off')
        fig.colorbar(im1, ax=axes[0])

        im2 = axes[1].imshow(original_blank, cmap='hot')
        axes[1].set_title(f'Blank Channel ({blank_name})')
        axes[1].axis('off')
        fig.colorbar(im2, ax=axes[1])

        im3 = axes[2].imshow(subtracted_marker, cmap='hot')
        axes[2].set_title(f'Subtracted {VERIFY_MARKER}')
        axes[2].axis('off')
        fig.colorbar(im3, ax=axes[2])

        plt.suptitle(f'Verification for marker: {VERIFY_MARKER}')
        plt.tight_layout(rect=(0, 0.03, 1, 0.95))
        plt.show()
        
        px, py = (y2-y1)//2, (x2-x1)//2
        orig_val = original_marker[px, py]
        blank_val = original_blank[px, py]
        sub_val = subtracted_marker[px, py]
        print(f"\n--- Single Pixel Check at ({py}, {px}) ---")
        print(f"Original Value: {orig_val}")
        print(f"Blank Value:    {blank_val}")
        print(f"Subtracted Value: {sub_val}")
        print(f"Expected Value:   {max(0, orig_val - blank_val)}")
        print("-------------------------------------")

    except Exception as e:
        log_time(f"Verification failed: {e}")
else:
    log_time("Skipping verification: Output file not found or marker not specified.")

log_time("Verification step finished.")